# 人工智慧概念

## 📌 學習目標

完成本 Notebook 後，你將能夠：

1. 說明人工智慧的基本定義與常見類型。
2. 區分分析型 AI、預測型 AI 與生成型 AI 的應用目的。
3. 辨識醫療、金融、製造、交通與娛樂等領域中的 AI 應用。
4. 理解資料蒐集、資料清洗與資料轉換在 AI 專案中的角色。
5. 使用 Python 進行簡單的資料分析、預測與文字生成概念示範。

本章重點不是訓練大型模型，而是透過輕量範例理解 AI 如何從資料中找出模式、做出預測，並產生簡單內容。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節所需的 Python 套件，並準備後續範例會使用的顯示設定。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import re

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.unicode_minus'] = False

print('環境設定完成')


## 核心概念說明

人工智慧是一種讓機器模擬人類智慧的技術，使電腦能夠執行學習、推理、分類、預測、感知與內容產生等任務。

依照功能目的，可以將常見 AI 應用分為三類：

| 類型 | 主要目的 | 例子 |
|---|---|---|
| 分析型 AI | 從資料中找出模式與洞察 | 客戶分群、銷售分析、異常偵測 |
| 預測型 AI | 根據歷史資料預測未來結果 | 銷售預測、信用風險評估、設備故障預測 |
| 生成型 AI | 根據提示產生文字、圖片、語音或影片等內容 | 文章草稿、客服回覆、簡報大綱 |

在實務上，AI 專案通常不會一開始就訓練模型，而是先處理資料。資料品質會直接影響 AI 系統的可靠性，因此資料蒐集、資料清洗與資料轉換是 AI 應用規劃中的重要基礎。


In [ ]:
# ── 示範：分析型 AI 的資料洞察 ─────────────────────────
# 這段程式碼使用簡單的交易資料，示範分析型 AI 如何從資料中找出不同客群的消費模式。

import pandas as pd
import matplotlib.pyplot as plt

customers = pd.DataFrame({
    'customer_id': ['C001', 'C002', 'C003', 'C004', 'C005', 'C006'],
    'age': [22, 35, 47, 29, 41, 53],
    'monthly_visits': [8, 3, 2, 6, 4, 1],
    'avg_spending': [450, 1200, 1800, 700, 1500, 2200]
})

def segment_customer(row):
    if row['avg_spending'] >= 1500:
        return '高價值客戶'
    elif row['monthly_visits'] >= 6:
        return '高互動客戶'
    else:
        return '一般客戶'

customers['segment'] = customers.apply(segment_customer, axis=1)
summary = customers.groupby('segment')[['monthly_visits', 'avg_spending']].mean().round(1)

print('客戶資料：')
print(customers)
print('\n各客群平均表現：')
print(summary)

summary['avg_spending'].plot(kind='bar', color=['#4C78A8', '#F58518', '#54A24B'])
plt.title('不同客群的平均消費金額')
plt.xlabel('客群')
plt.ylabel('平均消費金額')
plt.xticks(rotation=0)
plt.show()


## 資料處理與分析流程

AI 專案常見的資料準備流程包含：

1. **資料蒐集**：取得結構化、半結構化或非結構化資料，例如資料庫表格、JSON、CSV、圖片、音訊或文字。
2. **資料清洗**：處理遺缺值、重複值、錯誤值與離群值，提升資料品質。
3. **資料轉換**：將資料轉成適合分析或模型使用的格式，例如文字轉數值、類別轉編碼、數值正規化。
4. **資料分析或建模**：依照任務選擇分析、預測或生成方法。

在 iPAS 考試情境中，應特別注意 AI 不只是模型本身，還包含資料、演算法、應用開發與實際導入場景。


In [ ]:
# ── 示範：資料清洗與轉換 ──────────────────────────────
# 這段程式碼示範如何處理遺缺值、重複值與錯誤值，並將文字類別轉換成可分析的數值欄位。

import pandas as pd
import numpy as np

raw_data = pd.DataFrame({
    'user_id': [1, 2, 2, 3, 4, 5],
    'age': [25, np.nan, np.nan, -3, 45, 31],
    'industry': ['醫療', '金融', '金融', '製造', '交通', '娛樂'],
    'uses_ai': ['yes', 'no', 'no', 'yes', 'yes', 'no']
})

print('原始資料：')
print(raw_data)

cleaned = raw_data.drop_duplicates(subset=['user_id']).copy()
cleaned.loc[cleaned['age'] < 0, 'age'] = np.nan
cleaned['age'] = cleaned['age'].fillna(cleaned['age'].median())
cleaned['uses_ai_binary'] = cleaned['uses_ai'].map({'yes': 1, 'no': 0})
industry_encoded = pd.get_dummies(cleaned['industry'], prefix='industry')
cleaned = pd.concat([cleaned, industry_encoded], axis=1)

print('\n清洗與轉換後資料：')
print(cleaned)
print('\n平均年齡：', round(cleaned['age'].mean(), 1))
print('AI 使用比例：', round(cleaned['uses_ai_binary'].mean(), 2))


## 預測型 AI 的簡易模型

以下用線性迴歸示範預測型 AI：從歷史廣告投入與銷售量的關係，估計未來預算下的銷售量。


In [ ]:
# ── 實際應用：預測型 AI 的簡易模型 ───────────────────────
# 這段程式碼使用線性迴歸示範預測型 AI 的概念：根據歷史廣告投入預測銷售量。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

ad_budget = np.array([10, 20, 30, 40, 50, 60, 70, 80]).reshape(-1, 1)
sales = np.array([22, 28, 35, 41, 48, 55, 61, 68])

model = LinearRegression()
model.fit(ad_budget, sales)

future_budget = np.array([[90], [100]])
predicted_sales = model.predict(future_budget)

print('模型斜率：', round(model.coef_[0], 2))
print('模型截距：', round(model.intercept_, 2))
for budget, pred in zip(future_budget.flatten(), predicted_sales):
    print(f'廣告預算 {budget} 萬元，預測銷售量約 {pred:.1f} 單位')

plt.scatter(ad_budget, sales, label='歷史資料')
plt.plot(ad_budget, model.predict(ad_budget), color='red', label='預測趨勢線')
plt.scatter(future_budget, predicted_sales, color='green', label='未來預測')
plt.title('預測型 AI：廣告預算與銷售量')
plt.xlabel('廣告預算（萬元）')
plt.ylabel('銷售量')
plt.legend()
plt.show()


## 生成型 AI 的輕量示範

生成型 AI 會依照提示詞產生新內容。以下不使用大型語言模型，改用詞頻統計做輕量示範：先依提示詞挑出語料中相關的句子，再從中取出關鍵詞組成一段簡短回覆。


In [ ]:
# ── 示範：生成型 AI 的輕量文字概念 ───────────────────────
# 這段程式碼不用大型語言模型，而是用詞頻統計示範生成型 AI 的核心概念：
# 先依輸入提示挑出相關語料，再從中取出關鍵詞產生簡短文字。

from collections import Counter

corpus = [
    '人工智慧 可以 協助 醫療 診斷 與 影像 分析',
    '人工智慧 可以 協助 金融 風險 評估 與 詐欺 偵測',
    '人工智慧 可以 協助 製造 品質 檢測 與 預測性 維護',
    '生成型 AI 可以 根據 提示 產生 文字 圖像 與 摘要'
]

prompt = '請說明人工智慧在企業中的用途'

# 用「提示詞與句子的共同字元數」當作相關度，模擬檢索步驟；真實模型會改用語意向量比對。
def relevance(sentence, prompt):
    return len(set(sentence.replace(' ', '')) & set(prompt))

ranked = sorted(corpus, key=lambda sentence: relevance(sentence, prompt), reverse=True)
related_sentences = [s for s in ranked if relevance(s, prompt) > 0][:3]

related_words = []
for sentence in related_sentences:
    related_words.extend(sentence.split())

# 略過連接性質的詞，以及提示詞裡已經出現過的詞，留下最能代表主題的關鍵詞
function_words = {'可以', '協助', '與', '根據'}
keywords = [
    word for word, _ in Counter(related_words).most_common()
    if word not in function_words and word not in prompt
][:5]
generated = '、'.join(keywords)

print('提示詞：', prompt)
print('挑出的相關語料：')
for sentence in related_sentences:
    print(' -', sentence)
print('根據相關語料產生的關鍵概念：', generated)
print('簡短生成結果：人工智慧可用於' + generated + '等任務，協助企業提升效率與決策品質。')


## 🧪 自我測驗

請完成下方 TODO 填空，實作資料清洗（去重、錯誤值轉缺失、缺失值填補）與 AI 類型分類。


In [ ]:
# ── 🧪 自我測驗 ──────────────────────────────────
# 請完成下方 TODO 填空，實作資料清洗與 AI 類型分類功能，並確認輸出符合 Expected 註解。

import pandas as pd
import numpy as np

records = pd.DataFrame({
    'case_id': [101, 102, 102, 103, 104],
    'task': ['找出客戶分群', '預測下月銷售', '預測下月銷售', '產生客服回覆', '檢查交易異常'],
    'value': [80, np.nan, np.nan, 120, -999]
})

# TODO 1: 依 case_id 移除重複資料，保留第一筆
cleaned = records.drop_duplicates(subset=['case_id']).copy()

# Expected: 清洗後資料筆數為 4
print('清洗後資料筆數：', len(cleaned))

# TODO 2: 將明顯錯誤值 -999 改成缺失值 np.nan
cleaned.loc[cleaned['value'] == -999, 'value'] = np.nan

# TODO 3: 使用 value 欄位的中位數填補缺失值
cleaned['value'] = cleaned['value'].fillna(cleaned['value'].median())

# Expected: value 欄位為 [80.0, 100.0, 120.0, 100.0]
print('填補後 value：', cleaned['value'].tolist())

def classify_ai_type(task):
    if '預測' in task:
        return '預測型 AI'
    if '產生' in task or '生成' in task:
        return '生成型 AI'
    return '分析型 AI'

# TODO 4: 依 task 內容分類 AI 類型
cleaned['ai_type'] = cleaned['task'].apply(classify_ai_type)

# Expected: AI 類型分類為 ['分析型 AI', '預測型 AI', '生成型 AI', '分析型 AI']
print('AI 類型分類：', cleaned['ai_type'].tolist())

print('\n完成結果：')
print(cleaned)
